# Swaps con Boeing (BA): tasas, divisas y equity swaps

Cuaderno para **Google Colab** basado en el **Capítulo 7 (Swaps)** de *Options, Futures, and Other Derivatives* (Hull, 11.ª ed.), personalizado para **Boeing (BA)**.
Los datos de precios vienen de Yahoo Finance; las **tasas, spreads y posiciones son hipotéticos** y están marcados para reemplazarlos con datos de **LSEG** o **TradingView** (ver sección 6).

| Sección | Contenido |
|---|---|
| 1 | Swap de tasa fija contra variable (SOFR): flujos, valuación (bonos y FRAs), tasa par, DV01 |
| 2 | Ventaja comparativa: Boeing como emisor de crédito más débil frente a un emisor de mejor calidad |
| 3 | Swap de divisas USD/MXN: flujos, valuación y tipo de cambio de equilibrio |
| 4 | **Equity swaps sobre BA:** (A) BA contra tasa fija, (B) BA contra SPY, (C) BA contra el ETF de aeroespacial y defensa ITA. Flujos reales, valuación a mitad de vida y riesgo |
| 5 | **Cobertura de una posición en BA** con un equity swap sobre BA o sobre SPY (cobertura por beta) |
| 6 | Resumen y **datos que debes conseguir** |

**Cómo correrlo:** *Entorno de ejecución → Ejecutar todo*. Si Yahoo no responde para un activo, ese activo usa datos **simulados** (anclados a un precio de respaldo que debes actualizar) y lo avisa.
Material educativo; no es recomendación de inversión.

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq

pd.options.display.float_format = '{:,.4f}'.format
plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True})

# ================== PARÁMETROS (cámbialos aquí) ==================
TICKERS = ["BA", "SPY", "ITA"]                   # BA = Boeing; SPY = mercado; ITA = ETF de aeroespacial y defensa
YF = {t: t for t in TICKERS}
DIV_YIELD = {"BA": 0.0, "SPY": 0.011, "ITA": 0.008}   # CONFIRMA en LSEG (BA no paga dividendo desde 2020; ITA es aproximado)
COLORES = {"BA": "navy", "SPY": "royalblue", "ITA": "seagreen"}

SOFR_ON = 0.0397                # SOFR overnight, fijing del 3-nov-2026 (LSEG: USDSOFR=)
R_FREE = 0.040553               # OIS 3M (LSEG USDSROIS=, 8-nov-2026): referencia a 3 meses para financiamiento y cobertura
FIN_SPREAD = 0.0050             # spread sobre SOFR que paga el receptor del rendimiento (HIPOTÉTICO: pide cotización a la mesa)
N_EQ = 10e6                     # nocional de los equity swaps (USD)
N_TASAS = 100e6                 # nocional del swap de tasas (USD)

# Curva SOFR OIS real (LSEG USDSROIS=, bid, 8-nov-2026), tasas par en %. Se convierte a cero continuo (bootstrap).
OIS = {0.5: 4.1605, 1.0: 4.3215, 1.5: 4.3460, 2.0: 4.3756, 3.0: 4.3702}
T_CURVA = np.arange(0.5, 3.01, 0.5)
_r = {t: v / 100 for t, v in OIS.items()}
_df = {0.5: 1 / (1 + _r[0.5] * 0.5), 1.0: 1 / (1 + _r[1.0])}                        # simples hasta 1 año
_df[1.5] = (1 - _r[1.5] * _df[1.0] * 1.0) / (1 + _r[1.5] * 0.5)                      # cupón anual + stub
_df[2.0] = (1 - _r[2.0] * _df[1.0]) / (1 + _r[2.0])
_z2, _z3 = -np.log(_df[2.0]) / 2.0, None
_df[3.0] = (1 - _r[3.0] * (_df[1.0] + _df[2.0])) / (1 + _r[3.0])
_z3 = -np.log(_df[3.0]) / 3.0
_df[2.5] = np.exp(-(_z2 + 0.5 * (_z3 - _z2)) * 2.5)                                  # interpolación lineal de la tasa cero
Z_CURVA = np.array([-np.log(_df[t]) / t for t in T_CURVA])
print("Curva cero SOFR OIS (cap. continua):", {t: f"{z:.3%}" for t, z in zip(T_CURVA, Z_CURVA)})
TASA_FIJA_SWAP = 0.0430         # tasa fija pactada en el swap de tasas (HIPOTÉTICA; la tasa par sale de la curva)

# Posición de Boeing a cubrir (sección 5). HIPOTÉTICA: pon tus acciones reales
ACC_BA = 1000

# Respaldo si Yahoo no responde. >>> PLACEHOLDERS: ACTUALÍZALOS con el último precio de cada activo <<<
FALLBACK_PRICE = {"BA": 173.40, "SPY": 754.35, "ITA": 150.0}
FALLBACK_VOL = {"BA": 0.35, "SPY": 0.18, "ITA": 0.22}
FALLBACK_USDMXN = 18.50         # HIPOTÉTICO: reemplaza con el tipo de cambio actual
SEED = 7

## 1. Swap de tasa de interés: fija contra variable
En el swap *plain vanilla* una parte **paga tasa fija** y **recibe variable (SOFR)** sobre un nocional que **no se intercambia**; solo se pagan los flujos netos.
Aquí la tasa variable esperada de cada periodo es la **tasa forward** implícita en la curva cero. Dos enfoques equivalentes (Hull 7.7):
- **Bonos:** swap = bono variable − bono fijo (el bono variable vale el nocional justo tras un reajuste).
- **FRAs:** swap = serie de FRAs valuados a la tasa forward.

In [ ]:
dt = 0.5
Z = np.exp(-Z_CURVA * T_CURVA)                       # factores de descuento
Z_ant = np.r_[1.0, Z[:-1]]
fwd = (Z_ant / Z - 1) / dt                           # forwards semestrales (anualizadas)

tasa_par = (1 - Z[-1]) / (dt * Z.sum())             # tasa fija que hace el swap valer 0 hoy

def valor_swap_tasas(shift_pb=0.0, tasa_fija=TASA_FIJA_SWAP):
    """Valor para el que PAGA fija y RECIBE variable (enfoque de bonos)."""
    Zs = np.exp(-(Z_CURVA + shift_pb / 1e4) * T_CURVA)
    B_fijo = (N_TASAS * tasa_fija * dt * Zs).sum() + N_TASAS * Zs[-1]
    return N_TASAS - B_fijo

V_bonos = valor_swap_tasas()
V_fra = (N_TASAS * dt * (fwd - TASA_FIJA_SWAP) * Z).sum()
dv01 = (valor_swap_tasas(+0.5) - valor_swap_tasas(-0.5))        # cambio por +1 pb (diferencia central de ±0,5 pb)

flujos = pd.DataFrame({"t (años)": T_CURVA, "Tasa cero": Z_CURVA, "Forward (var. esperada)": fwd,
                       "Pago fijo": N_TASAS * TASA_FIJA_SWAP * dt, "Cobro variable": N_TASAS * fwd * dt})
flujos["Neto paga fijo"] = flujos["Cobro variable"] - flujos["Pago fijo"]
print(flujos.to_string(formatters={"Pago fijo": "{:,.0f}".format, "Cobro variable": "{:,.0f}".format, "Neto paga fijo": "{:,.0f}".format}))
print(f"\nTasa swap par: {tasa_par:.4%}   |   tasa fija pactada: {TASA_FIJA_SWAP:.4%}")
print(f"Valor (bonos), paga fija:  USD {V_bonos:,.0f}   |   Valor (FRAs): USD {V_fra:,.0f}   |   coinciden: {np.isclose(V_bonos, V_fra)}")
print(f"DV01 del pagador de fija: USD {dv01:,.0f} por +1 pb")

desp = np.arange(-100, 101, 25)
vals = [valor_swap_tasas(s) / 1e6 for s in desp]
fig, ax = plt.subplots()
ax.plot(desp, vals, marker="o"); ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("Desplazamiento paralelo de la curva (pb)"); ax.set_ylabel("Valor para el pagador de fija (millones USD)")
ax.set_title("Sensibilidad del swap de tasas")
plt.show()
print("Interpretación: si la fija pactada es menor que la tasa par, el pagador de fija tiene un swap con valor positivo; "
      "gana cuando suben las tasas y pierde cuando bajan.")

## 2. Ventaja comparativa con Boeing (BBB- / Baa3)
Si los diferenciales de crédito en **fija** y en **variable** no son iguales, un swap puede abaratar a ambas partes.
- **Boeing (B):** calificación **BBB-** (S&P, estable; Fitch, perspectiva positiva) y **Baa3** (Moody's, en la tabla de bonos). Sus rendimientos de mercado son **reales** (tabla de bonos, 6-nov-2026).
- **A:** emisor de mejor calidad (por ejemplo RTX o Lockheed; A-). **Sus spreads son hipotéticos** hasta que consigas sus bonos.
- El spread de Boeing en **variable** (SOFR + x) también es **hipotético**: el reporte 2025 no da el margen de las líneas de crédito.
Un intermediario se queda con un margen.

In [ ]:
# ---------- Bonos reales de Boeing (YTW y precio al 6-nov-2026, capturas de bonos) ----------
HOY = pd.Timestamp("2026-11-08")
BONOS = [  # (vencimiento, cupón %, precio %, YTW %)
    ("2027-02-01", 2.700, 99.45, 5.05), ("2027-05-01", 5.040, 99.96, 5.20), ("2027-05-01", 6.259, 100.61, 4.73),
    ("2028-02-01", 3.250, 97.92, 5.00), ("2028-03-01", 3.250, 97.53, 5.21), ("2028-11-01", 3.450, 96.87, 5.13),
    ("2029-03-01", 3.200, 95.74, 5.17), ("2029-05-01", 6.298, 102.37, 5.25),
    ("2030-02-01", 2.950, 92.96, 5.34), ("2030-05-01", 5.150, 99.35, 5.36),
    ("2031-02-01", 3.625, 93.29, 5.42), ("2031-05-01", 6.388, 103.73, 5.41)]
OIS_PTS = {**OIS, 4.0: 4.3653, 5.0: 4.3637}                     # tasas par OIS (%) de LSEG
tk = np.array(sorted(OIS_PTS)); vk = np.array([OIS_PTS[t] for t in tk])
bon = pd.DataFrame(BONOS, columns=["Vence", "Cupón %", "Precio %", "YTW %"])
bon["Años"] = (pd.to_datetime(bon["Vence"]) - HOY).dt.days / 365.25
bon["OIS %"] = np.interp(bon["Años"], tk, vk)
bon["Spread vs OIS (pb)"] = (bon["YTW %"] - bon["OIS %"]) * 100
print(bon.round(2).to_string(index=False))

tramo3 = bon[(bon["Años"] > 2.0) & (bon["Años"] < 3.5)]; tramo5 = bon[bon["Años"] >= 3.5]
Y3, SP3 = tramo3["YTW %"].mean() / 100, tramo3["Spread vs OIS (pb)"].mean()
Y5, SP5 = tramo5["YTW %"].mean() / 100, tramo5["Spread vs OIS (pb)"].mean()
print(f"\nBoeing ~3 años (vencen 2029-30): rendimiento promedio {Y3:.2%}  →  spread ≈ {SP3:.0f} pb sobre OIS")
print(f"Boeing ~5 años (vencen 2031):     rendimiento promedio {Y5:.2%}  →  spread ≈ {SP5:.0f} pb sobre OIS")
print("Nota: el spread es contra la curva OIS; los bonos cotizan en base semestral, así que es una aproximación.")

In [ ]:
# ---------- Ventaja comparativa a 3 años ----------
OIS3 = OIS_PTS[3.0] / 100
fija_B, spread_B = Y3, 0.0065         # B = Boeing: fija = rendimiento REAL a ~3 años | variable SOFR + 0.65 % (HIPOTÉTICO)
fija_A, spread_A = OIS3 + 0.0033, 0.0025   # A = emisor A-: fija = OIS 3A + 33 pb | variable SOFR + 0.25 % (AMBOS HIPOTÉTICOS)
margen_banco = 0.0004                 # el intermediario cobra 4 pb

dif_fija, dif_var = fija_B - fija_A, spread_B - spread_A
ganancia_total = dif_fija - dif_var
g_neta = (ganancia_total - margen_banco) / 2          # ganancia de cada parte tras el margen

costo_A = spread_A - g_neta                           # sobre SOFR
costo_B = fija_B - g_neta                             # tasa fija efectiva

print(f"Fija A: {fija_A:.2%} | fija B (Boeing): {fija_B:.2%}")
print(f"Diferencia en fija: {dif_fija:.2%} | en variable: {dif_var:.2%} | ganancia total: {ganancia_total:.2%}")
print(f"Margen del intermediario: {margen_banco:.2%}  →  ganancia neta de cada parte: {g_neta:.3%}")
print(f"Costo final de A: SOFR {costo_A:+.3%}   (sin swap: SOFR {spread_A:+.2%})")
print(f"Costo final de B (Boeing): {costo_B:.3%} fijo   (sin swap: {fija_B:.2%})")
print("Cautela (Hull): la ventaja suele reflejar que el diferencial en variable es de corto plazo; Boeing sigue asumiendo riesgo de crédito. No es un almuerzo gratis. "
      "Si la ganancia total sale negativa o menor al margen, no hay swap que convenga.")

## 3. Swap de divisas fijo-fijo (USD / MXN)
Se **intercambian principales** al inicio y al final, e intereses en cada moneda. Valuación (Hull 7.10): $V = B_{USD} - B_{MXN}/S_0$ con $S_0$ en MXN por USD.
Caso: empresa mexicana con deuda en USD e ingresos en MXN: **recibe USD y paga MXN**. Las tasas MXN y el tipo de cambio son **hipotéticos**: reemplázalos con datos de LSEG/TradingView.

In [ ]:
def bajar_usdmxn():
    try:
        import yfinance as yf
        s = yf.download("MXN=X", period="1mo", progress=False, auto_adjust=True)["Close"]
        s = s.iloc[:, 0] if isinstance(s, pd.DataFrame) else s
        s = s.dropna()
        if len(s):
            return float(s.iloc[-1]), "Yahoo Finance"
    except Exception:
        pass
    return FALLBACK_USDMXN, "supuesto (respaldo)"

S0_FX, fuente_fx = bajar_usdmxn()
print(f"USD/MXN spot: {S0_FX:.4f}  ({fuente_fx})")

N_usd = 10e6; N_mxn = N_usd * S0_FX
cup_usd, cup_mxn = 0.0400, 0.0875          # HIPOTÉTICO: tasas fijas del swap
r_usd_d, r_mxn_d = R_FREE, 0.0900          # HIPOTÉTICO: tasas de descuento planas (continuas)

flujos_div = pd.DataFrame({"Año": [0, 1, 2, 3],
    "USD": [-N_usd, N_usd * cup_usd, N_usd * cup_usd, N_usd * cup_usd + N_usd],
    "MXN": [N_mxn, -N_mxn * cup_mxn, -N_mxn * cup_mxn, -N_mxn * cup_mxn - N_mxn]})
print(flujos_div.to_string(formatters={"USD": "{:,.0f}".format, "MXN": "{:,.0f}".format}))

def valor_swap_divisas(S, anios_restantes=2):
    t = np.arange(1, anios_restantes + 1)
    cf_usd = np.full(len(t), N_usd * cup_usd); cf_usd[-1] += N_usd
    cf_mxn = np.full(len(t), N_mxn * cup_mxn); cf_mxn[-1] += N_mxn
    B_usd = (cf_usd * np.exp(-r_usd_d * t)).sum()
    B_mxn = (cf_mxn * np.exp(-r_mxn_d * t)).sum()
    return B_usd - B_mxn / S

print("\nValor del swap un año después (quedan 2 años), para quien recibe USD y paga MXN:")
for S in [S0_FX * 0.9, S0_FX, S0_FX * 1.1, S0_FX * 1.2]:
    print(f"  USD/MXN = {S:6.2f}  →  valor: USD {valor_swap_divisas(S):>12,.0f}")
try:
    S_eq = brentq(lambda s: valor_swap_divisas(s), 5, 60)
    print(f"Tipo de cambio que hace valer cero el swap: {S_eq:.3f} MXN/USD")
except Exception:
    pass

spots = np.linspace(S0_FX * 0.8, S0_FX * 1.3, 40)
fig, ax = plt.subplots()
ax.plot(spots, [valor_swap_divisas(s) / 1e6 for s in spots]); ax.axhline(0, color="k", lw=0.8); ax.axvline(S0_FX, color="gray", ls="--")
ax.set_xlabel("USD/MXN"); ax.set_ylabel("Valor del swap (millones USD)"); ax.set_title("Empresa que recibe USD y paga MXN")
plt.show()
print("Interpretación: si el peso se deprecia (sube USD/MXN), las obligaciones en MXN valen menos en USD y el swap GANA valor: es una cobertura natural de la deuda en dólares.")

## 4. Equity swaps sobre Boeing (BA)
El receptor cobra el **rendimiento total** de BA (precio + dividendos; BA hoy no paga dividendo) y paga:
- **Swap A:** SOFR + spread (tasa fija por periodo).
- **Swap B:** el rendimiento de **SPY** (apuesta relativa: BA contra el mercado).
- **Swap C:** el rendimiento de **ITA** (BA contra su sector, aeroespacial y defensa).
Los periodos son trimestrales (63 días hábiles) sobre el último año disponible.

In [ ]:
# ---------- Datos ----------
def bajar(t, years=2):
    try:
        import yfinance as yf
        fin = pd.Timestamp.today().normalize() + pd.Timedelta(days=1)
        df = yf.download(YF[t], start=fin - pd.DateOffset(years=years), end=fin, auto_adjust=True, progress=False, timeout=20)["Close"]
        df = df.iloc[:, 0] if isinstance(df, pd.DataFrame) else df
        df = df.dropna().astype(float)
        if len(df) >= 60:
            return df
    except Exception as e:
        print(f"  {t}: no se pudo descargar ({str(e)[:60]})")
    return None

def sintetico(t, n=500):
    rng = np.random.default_rng(SEED + TICKERS.index(t))
    common = np.random.default_rng(SEED).normal(0, 1, n)          # factor de mercado común para que haya correlación
    z = 0.6 * common + 0.8 * rng.normal(0, 1, n)
    s = FALLBACK_PRICE[t] * np.exp(np.cumsum(0.0002 + FALLBACK_VOL[t] / np.sqrt(252) * z))
    s = s / s[-1] * FALLBACK_PRICE[t]
    return pd.Series(s, index=pd.bdate_range(end=pd.Timestamp.today().normalize(), periods=n))

PRECIOS, SINT = {}, {}
for t in TICKERS:
    p = bajar(t); SINT[t] = p is None
    PRECIOS[t] = sintetico(t) if SINT[t] else p
S0 = {t: float(PRECIOS[t].iloc[-1]) for t in TICKERS}
R = {t: 100 * np.log(PRECIOS[t] / PRECIOS[t].shift(1)).dropna() for t in TICKERS}   # rendimientos diarios en %
print(pd.DataFrame({t: {"Fuente": "SIMULADO (actualiza los precios de respaldo)" if SINT[t] else "Yahoo Finance", "Obs.": len(PRECIOS[t]),
      "Desde": PRECIOS[t].index[0].date(), "Hasta": PRECIOS[t].index[-1].date(), "Último": round(S0[t], 2)} for t in TICKERS}).T.to_string())

# ---------- 4.1 Flujos de los tres swaps ----------
STEP = 63
def rend_periodos(t, n_per=4):
    s = PRECIOS[t]; pos = [len(s) - 1 - STEP * k for k in range(n_per, -1, -1)]
    obs = s.iloc[pos]
    return obs.pct_change().dropna().values + DIV_YIELD[t] * STEP / 252, obs

rBA, obsBA = rend_periodos("BA"); rSPY, _ = rend_periodos("SPY"); rITA, _ = rend_periodos("ITA")
fijo_p = (R_FREE + FIN_SPREAD) * STEP / 252
flujos = pd.DataFrame({"Rend. BA": rBA, "Rend. SPY": rSPY, "Rend. ITA": rITA,
    "Neto A: BA − (SOFR+spread)": N_EQ * (rBA - fijo_p), "Neto B: BA − SPY": N_EQ * (rBA - rSPY), "Neto C: BA − ITA": N_EQ * (rBA - rITA)},
    index=[f"{obsBA.index[i].date()} → {obsBA.index[i+1].date()}" for i in range(4)])
print()
print(flujos.to_string(formatters={"Rend. BA": "{:.2%}".format, "Rend. SPY": "{:.2%}".format, "Rend. ITA": "{:.2%}".format,
      "Neto A: BA − (SOFR+spread)": "{:,.0f}".format, "Neto B: BA − SPY": "{:,.0f}".format, "Neto C: BA − ITA": "{:,.0f}".format}))
print("\nTotales (USD):", {c: round(flujos[c].sum()) for c in flujos.columns if c.startswith("Neto")})

flujos[[c for c in flujos.columns if c.startswith("Neto")]].div(1e6).reset_index(drop=True).plot(kind="bar", title="Flujo neto del receptor de BA por trimestre (millones USD)")
plt.axhline(0, color="k", lw=0.8); plt.ylabel("Millones USD"); plt.xlabel("Trimestre"); plt.show()
print("Interpretación: en el Swap A el receptor gana cuando BA rinde más que SOFR + spread; en B y C solo gana si BA le gana al mercado o al sector, "
      "así que sus flujos dependen del rendimiento relativo, no de la dirección general del mercado.")

In [ ]:
# ---------- 4.2 Valuación a mitad de vida ----------
# Hipótesis: pasaron 2 trimestres y quedan 2. Receptor del rendimiento de BA:
#   contra fija:      V = N * S_t/S_0 − B_fijo          (Hull 7.13)
#   contra otro activo: V = N * (S_t/S_0 del BA − S_t/S_0 del otro)
def ratio(t, k_atras=2):
    s = PRECIOS[t]; s0 = float(s.iloc[len(s) - 1 - STEP * k_atras]); return S0[t] * np.exp(DIV_YIELD[t] * k_atras * STEP / 252) / s0, s0

tt = np.array([1, 2]) * STEP / 252
cup = N_EQ * (R_FREE + FIN_SPREAD) * STEP / 252
B_fijo = cup * np.exp(-R_FREE * tt[0]) + (cup + N_EQ) * np.exp(-R_FREE * tt[1])
rBA2, s0BA = ratio("BA"); rSPY2, _ = ratio("SPY"); rITA2, _ = ratio("ITA")
val = pd.DataFrame({
    "Swap": ["A: BA contra SOFR + spread", "B: BA contra SPY", "C: BA contra ITA"],
    "Valor para el receptor de BA (USD)": [N_EQ * rBA2 - B_fijo, N_EQ * (rBA2 - rSPY2), N_EQ * (rBA2 - rITA2)]})
val["% del nocional"] = val["Valor para el receptor de BA (USD)"] / N_EQ
print(f"BA hace 2 trimestres: {s0BA:,.2f} → hoy: {S0['BA']:,.2f}  (rend. acumulado {rBA2 - 1:+.2%})")
print(val.to_string(index=False, formatters={"Valor para el receptor de BA (USD)": "{:,.0f}".format, "% del nocional": "{:.2%}".format}))
print("\nInterpretación: valor positivo = al receptor de BA le conviene el swap (BA superó lo que debe pagar); negativo = va perdiendo.")

In [ ]:
# ---------- 4.3 Riesgo: volatilidad, beta y VaR ----------
LAM = 0.94
def ewma_var(u, lam=LAM):
    v = np.var(u[:30]); out = [v]
    for x in u[:-1]:
        v = lam * v + (1 - lam) * x ** 2; out.append(v)
    return np.array(out)

RR = pd.concat([R[t].rename(t) for t in TICKERS], axis=1).dropna()
X = RR.values; C = np.cov(X[:30].T)
for i in range(1, len(X)):
    C = LAM * C + (1 - LAM) * np.outer(X[i - 1], X[i - 1])          # cov EWMA (%², diaria)
for j in range(len(X)):
    pass
ix = {t: i for i, t in enumerate(TICKERS)}
vol_anual = {t: np.sqrt(252 * C[ix[t], ix[t]]) / 100 for t in TICKERS}
corr = lambda a, b: C[ix[a], ix[b]] / np.sqrt(C[ix[a], ix[a]] * C[ix[b], ix[b]])
beta_BA_SPY = C[ix["BA"], ix["SPY"]] / C[ix["SPY"], ix["SPY"]]
vol_rel = lambda a, b: np.sqrt(252 * (C[ix[a], ix[a]] + C[ix[b], ix[b]] - 2 * C[ix[a], ix[b]])) / 100
vol_resid = np.sqrt(252 * (C[ix["BA"], ix["BA"]] - beta_BA_SPY ** 2 * C[ix["SPY"], ix["SPY"]])) / 100

filas = {"Swap A: BA contra tasa fija": vol_anual["BA"], "Swap B: BA contra SPY": vol_rel("BA", "SPY"), "Swap C: BA contra ITA": vol_rel("BA", "ITA")}
riesgo = pd.DataFrame({"σ anual del flujo": filas}); riesgo["VaR 99% 10d (% nocional)"] = 2.326 * riesgo["σ anual del flujo"] * np.sqrt(10 / 252)
riesgo["VaR 99% 10d (USD)"] = N_EQ * riesgo["VaR 99% 10d (% nocional)"]
print(riesgo.to_string(formatters={"σ anual del flujo": "{:.1%}".format, "VaR 99% 10d (% nocional)": "{:.1%}".format, "VaR 99% 10d (USD)": "{:,.0f}".format}))
print(f"\nMuestra de {len(RR)} días. Correlación BA-SPY: {corr('BA','SPY'):.2f} | BA-ITA: {corr('BA','ITA'):.2f} | Beta de BA contra SPY: {beta_BA_SPY:.2f} | σ residual de BA (sin mercado): {vol_resid:.1%}")
print("Interpretación: el swap contra SPY o ITA elimina parte del riesgo de mercado/sector; lo que queda es el riesgo propio de Boeing "
      "(noticias de producción, entregas, certificaciones, resultados).")

## 5. Cobertura de una posición en Boeing
Tienes `ACC_BA` acciones de BA (hipotético). Dos formas de cubrir con equity swaps (pagas el rendimiento y recibes SOFR):
1. **Swap sobre BA** (cobertura total): casi elimina el riesgo de precio, pero también el potencial de ganancia.
2. **Swap sobre SPY por beta** (cobertura de mercado): neutraliza el riesgo sistemático y deja el riesgo propio de Boeing.

In [ ]:
N_cob = ACC_BA * S0["BA"]
H = 63 / 252                                       # horizonte de 3 meses
print(f"Posición: {ACC_BA:,.0f} acciones × {S0['BA']:,.2f} = USD {N_cob:,.0f} de nocional. Beta contra SPY: {beta_BA_SPY:.2f}\n")

# Volatilidad anual del P&L según la cobertura
vols = pd.Series({"Sin cobertura": vol_anual["BA"],
                  "Swap sobre SPY (β)": vol_resid,
                  "Swap sobre BA": 0.0}, name="σ anual del P&L")
print(vols.to_frame().to_string(formatters={"σ anual del P&L": "{:.1%}".format}))
print(f"VaR 99% a 10 días sin cobertura: USD {N_cob * 2.326 * vol_anual['BA'] * np.sqrt(10/252):,.0f}  |  con swap sobre SPY (β): USD {N_cob * 2.326 * vol_resid * np.sqrt(10/252):,.0f}")

# Escenarios a 3 meses: movimiento del mercado (SPY) y movimiento propio de BA (idiosincrático)
mercado = np.array([-0.10, -0.05, 0.0, 0.05, 0.10]); idio = np.array([-0.10, 0.0, 0.10])
carry = N_cob * R_FREE * H
filas = []
for m in mercado:
    for e in idio:
        x_ba = beta_BA_SPY * m + e
        sin = N_cob * x_ba
        con_spy = sin + beta_BA_SPY * N_cob * (R_FREE * H - m)      # paga rendimiento de SPY × β, recibe SOFR
        con_ba = sin + N_cob * (R_FREE * H - x_ba)                   # paga rendimiento de BA, recibe SOFR
        filas.append({"Mercado (SPY)": m, "Propio de BA": e, "Sin cobertura": sin, "Con swap sobre SPY (β)": con_spy, "Con swap sobre BA": con_ba})
esc = pd.DataFrame(filas)
print()
print(esc.to_string(index=False, formatters={"Mercado (SPY)": "{:+.0%}".format, "Propio de BA": "{:+.0%}".format,
      "Sin cobertura": "{:,.0f}".format, "Con swap sobre SPY (β)": "{:,.0f}".format, "Con swap sobre BA": "{:,.0f}".format}))

fig, ax = plt.subplots(figsize=(8, 4))
sub = esc[esc["Propio de BA"] == 0.0]
ax.plot(sub["Mercado (SPY)"] * 100, sub["Sin cobertura"] / 1e3, marker="o", label="Sin cobertura", color="gray")
ax.plot(sub["Mercado (SPY)"] * 100, sub["Con swap sobre SPY (β)"] / 1e3, marker="o", label="Swap sobre SPY (β)", color="royalblue")
ax.plot(sub["Mercado (SPY)"] * 100, sub["Con swap sobre BA"] / 1e3, marker="o", label="Swap sobre BA", color="navy")
ax.axhline(0, color="k", lw=0.8); ax.set_xlabel("Movimiento de SPY a 3 meses (%)"); ax.set_ylabel("P&L (miles USD)"); ax.legend()
ax.set_title("Cobertura de la posición en BA (sin shock propio de Boeing)"); plt.show()
print(f"Costo/beneficio: el swap recibe SOFR (~USD {carry:,.0f} a 3 meses) y paga el rendimiento; no hay prima, pero cedes el alza. "
      "El swap sobre SPY deja expuesto el riesgo propio de Boeing (columna 'Propio de BA').")

## 6. Resumen y datos que debes conseguir

**Ideas clave:** swap de tasas = bono variable − bono fijo = serie de FRAs; swap de divisas = bono en USD − bono en MXN convertido; equity swap = recibir/pagar el rendimiento de un activo contra una tasa u otro activo; para cubrir una posición, el swap sobre el propio activo elimina casi todo el riesgo y el swap sobre el mercado (por beta) elimina solo la parte sistemática.

**Datos para sustituir los supuestos hipotéticos**

| Dato | Para qué se usa | Dónde |
|---|---|---|
| ~~Curva SOFR OIS~~ (ya cargada: USDSROIS=, 8-nov-2026) | `OIS`, `Z_CURVA`, `R_FREE` | Actualiza cuando cambie la curva |
| ~~Bonos de Boeing y calificación~~ (ya cargados, 6-nov-2026). Falta: bonos de un comparable A- (RTX, Lockheed) y el margen variable de Boeing | Sección 2 | LSEG: Debt & Credit |
| Spread de financiamiento del equity swap (SOFR + x pb) y costo de préstamo de BA | `FIN_SPREAD` | Cotización de la mesa / prime broker |
| Dividend yield de BA, SPY e ITA | `DIV_YIELD` | LSEG (Overview) |
| Precio actual de BA, SPY e ITA (solo para el respaldo) | `FALLBACK_PRICE` | TradingView / LSEG |
| Número de acciones de BA y fecha de compra (si hay posición real) | `ACC_BA`, sección 5 | Tu cuenta |
| USD/MXN spot y curva MXN (TIIE / Mbono) | Sección 3 | LSEG / TradingView |

**Para practicar:** (1) sube `FIN_SPREAD` a 1.5 % y mira cómo cambia el Swap A; (2) cambia ITA por otro benchmark (por ejemplo, un ETF industrial); (3) usa un nocional de 5 y 20 millones y compara el VaR; (4) modifica `ACC_BA` y revisa el escenario de caída del mercado.

*Referencia:* J. C. Hull, *Options, Futures, and Other Derivatives*, 11.ª ed., Cap. 7. Material educativo.